In [1]:
# =============================================================
# Cell 1 — Task Registry & Dispatcher
# =============================================================
# We build a registry of claim-processing tasks, each with its
# own optimized prompt configuration. The dispatcher routes
# incoming requests to the right task based on what's needed.
# =============================================================
# Day 29: Multi-Task Architecture
# Phase 3, Notebook 9
# =============================================================

import boto3
import json
import time
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Any
from enum import Enum

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'


# ----- Utility -----
def ask(prompt, system=None, max_tokens=1024, temperature=0.0):
    """Call Bedrock and return response with metrics."""
    kwargs = {
        'modelId': model_id,
        'messages': [{'role': 'user', 'content': [{'text': prompt}]}],
        'inferenceConfig': {'maxTokens': max_tokens, 'temperature': temperature}
    }
    if system:
        kwargs['system'] = [{'text': system}]

    start = time.time()
    resp = bedrock.converse(**kwargs)
    latency = time.time() - start

    content = resp['output']['message'].get('content', [])
    text = content[0]['text'] if content and 'text' in content[0] else '[EMPTY RESPONSE]'

    return {
        'text':          text,
        'input_tokens':  resp['usage']['inputTokens'],
        'output_tokens': resp['usage']['outputTokens'],
        'latency':       round(latency, 2),
        'stop_reason':   resp.get('stopReason', 'unknown')
    }


# ----- Task Definition -----
class TaskType(Enum):
    """The types of tasks our system can perform."""
    CLASSIFY     = 'classify'
    FRAUD_SCREEN = 'fraud_screen'
    SEVERITY     = 'severity'
    COVERAGE     = 'coverage'
    COMMUNICATE  = 'communicate'


@dataclass
class TaskConfig:
    """Configuration for a single task in the registry."""
    task_type: TaskType
    name: str
    description: str
    system_prompt: str
    user_template: str           # must contain {claim_text}, may use {context}
    output_format: str = 'text'  # 'text' or 'json'
    max_tokens: int = 1024
    depends_on: List[TaskType] = field(default_factory=list)  # tasks that must run first


# ----- The Task Registry -----
TASK_REGISTRY: Dict[TaskType, TaskConfig] = {}


def register_task(config: TaskConfig):
    """Add a task to the registry."""
    TASK_REGISTRY[config.task_type] = config
    return config


# ----- Register our five core tasks -----

register_task(TaskConfig(
    task_type=TaskType.CLASSIFY,
    name='Claim Classification',
    description='Determine the claim type and coverage line',
    system_prompt=(
        'Senior P&C claims classifier. Determine claim type accurately.\n'
        'Rules: Be precise. When multiple types apply, pick the primary.'
    ),
    user_template=(
        'Classify this insurance claim.\n\n'
        '{claim_text}\n\n'
        'Respond in JSON only:\n'
        '{{"claim_type": "auto|homeowners|liability|commercial",'
        ' "sub_type": "string",'
        ' "confidence": "HIGH|MEDIUM|LOW",'
        ' "reasoning": "one sentence"}}'
    ),
    output_format='json',
    max_tokens=256,
    depends_on=[]
))

register_task(TaskConfig(
    task_type=TaskType.SEVERITY,
    name='Severity Assessment',
    description='Rate claim urgency and complexity',
    system_prompt=(
        'Claims severity analyst. Assess urgency based on injury, '
        'dollar amount, complexity, and litigation risk.'
    ),
    user_template=(
        'Assess the severity of this claim.\n\n'
        '{claim_text}\n\n'
        'Prior analysis:\n{context}\n\n'
        'Respond in JSON only:\n'
        '{{"severity": "LOW|MEDIUM|HIGH|CRITICAL",'
        ' "priority_score": 1-10,'
        ' "factors": ["string"],'
        ' "sla_hours": number}}'
    ),
    output_format='json',
    max_tokens=256,
    depends_on=[TaskType.CLASSIFY]
))

register_task(TaskConfig(
    task_type=TaskType.FRAUD_SCREEN,
    name='Fraud Screening',
    description='Check for fraud indicators and red flags',
    system_prompt=(
        'Insurance fraud analyst with SIU experience.\n'
        'Rules: Flag indicators objectively. Never accuse — only flag.\n'
        'Consider: timing, consistency, documentation gaps, '
        'claimant history patterns, and industry red flags.'
    ),
    user_template=(
        'Screen this claim for fraud indicators.\n\n'
        '{claim_text}\n\n'
        'Prior analysis:\n{context}\n\n'
        'Respond in JSON only:\n'
        '{{"fraud_risk": "LOW|MEDIUM|HIGH",'
        ' "risk_score": 1-10,'
        ' "indicators": ["string"],'
        ' "recommendation": "CLEAR|MONITOR|INVESTIGATE",'
        ' "reasoning": "string"}}'
    ),
    output_format='json',
    max_tokens=512,
    depends_on=[TaskType.CLASSIFY]
))

register_task(TaskConfig(
    task_type=TaskType.COVERAGE,
    name='Coverage Determination',
    description='Determine what is covered under the policy',
    system_prompt=(
        'Senior coverage analyst. Determine applicable coverage.\n'
        'Rules: No dollar settlement recommendations. Reference policy '
        'provisions by name. Flag coverage questions for supervisor review.'
    ),
    user_template=(
        'Determine coverage for this claim.\n\n'
        '{claim_text}\n\n'
        'Prior analysis:\n{context}\n\n'
        'Respond in JSON only:\n'
        '{{"primary_coverage": "string",'
        ' "applicable_provisions": ["string"],'
        ' "exclusions_to_check": ["string"],'
        ' "deductible_applies": true|false,'
        ' "subrogation_potential": true|false,'
        ' "coverage_issues": ["string"],'
        ' "recommendation": "APPROVE|REVIEW|DENY"}}'
    ),
    output_format='json',
    max_tokens=512,
    depends_on=[TaskType.CLASSIFY, TaskType.SEVERITY, TaskType.FRAUD_SCREEN]
))

register_task(TaskConfig(
    task_type=TaskType.COMMUNICATE,
    name='Policyholder Communication',
    description='Draft clear communication to the policyholder',
    system_prompt=(
        'Insurance customer communications specialist.\n'
        'Rules: Empathetic, clear, no jargon. Explain next steps.\n'
        'Never disclose internal fraud assessments or reserve estimates.\n'
        'Keep under 200 words.'
    ),
    user_template=(
        'Draft a message to the policyholder about their claim.\n\n'
        'Claim details:\n{claim_text}\n\n'
        'Internal analysis (DO NOT share with policyholder):\n{context}\n\n'
        'Write a professional, empathetic acknowledgment email that:\n'
        '1. Confirms receipt of the claim\n'
        '2. Summarizes what we understand happened\n'
        '3. Explains the next steps\n'
        '4. Provides a timeline expectation'
    ),
    output_format='text',
    max_tokens=512,
    depends_on=[TaskType.CLASSIFY, TaskType.COVERAGE]
))


# ----- The Dispatcher -----
class TaskDispatcher:
    """Routes claims to the appropriate task and executes it."""
    
    def __init__(self, registry: Dict[TaskType, TaskConfig]):
        self.registry = registry
    
    def get_task(self, task_type: TaskType) -> TaskConfig:
        """Look up a task configuration."""
        if task_type not in self.registry:
            raise ValueError(f"Task '{task_type.value}' not found in registry")
        return self.registry[task_type]
    
    def check_dependencies(self, task_type: TaskType, 
                           completed: Dict[TaskType, Any]) -> bool:
        """Verify all dependencies have been completed."""
        task = self.get_task(task_type)
        missing = [d for d in task.depends_on if d not in completed]
        if missing:
            print(f"  ⚠ Missing dependencies for {task_type.value}: "
                  f"{[m.value for m in missing]}")
            return False
        return True
    
    def build_context(self, task_type: TaskType, 
                      completed: Dict[TaskType, Any]) -> str:
        """Build context string from completed task results."""
        task = self.get_task(task_type)
        if not task.depends_on:
            return "No prior analysis available."
        
        context_parts = []
        for dep in task.depends_on:
            if dep in completed:
                result = completed[dep]
                context_parts.append(
                    f"[{dep.value.upper()}]: {result['text']}"
                )
        
        return "\n".join(context_parts) if context_parts else "No prior analysis available."
    
    def execute(self, task_type: TaskType, claim_text: str,
                completed: Dict[TaskType, Any] = None) -> Dict:
        """Execute a single task with dependency checking and context."""
        completed = completed or {}
        task = self.get_task(task_type)
        
        # Check dependencies
        if not self.check_dependencies(task_type, completed):
            return {'error': 'Missing dependencies', 'task': task_type.value}
        
        # Build context from prior results
        context = self.build_context(task_type, completed)
        
        # Render prompt
        prompt = task.user_template.format(
            claim_text=claim_text,
            context=context
        )
        
        # Execute
        result = ask(prompt, system=task.system_prompt, 
                    max_tokens=task.max_tokens)
        
        # Parse JSON if expected
        if task.output_format == 'json':
            try:
                raw = result['text'].strip()
                if raw.startswith("```"):
                    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
                result['parsed'] = json.loads(raw)
            except (json.JSONDecodeError, IndexError):
                result['parsed'] = None
                result['parse_error'] = True
        
        return result


# ----- Demo the Registry and Dispatcher -----
print("=" * 65)
print("Day 29: Multi-Task Architecture")
print("Task Registry & Dispatcher")
print("=" * 65)

print(f"\nRegistered Tasks: {len(TASK_REGISTRY)}\n")

dispatcher = TaskDispatcher(TASK_REGISTRY)

for task_type, config in TASK_REGISTRY.items():
    deps = [d.value for d in config.depends_on] if config.depends_on else ['none']
    print(f"  {config.task_type.value:<15} │ {config.name:<28} │ "
          f"format={config.output_format:<4} │ depends={deps}")

# ----- Run a single task to test -----
print(f"\n{'=' * 65}")
print("DISPATCHER TEST: Single Task (Classification)")
print(f"{'=' * 65}")

test_claim = (
    "Rear-end collision on Highway 401. Three vehicles involved. "
    "Claimant reports whiplash and vehicle damage to rear bumper and trunk. "
    "Police report filed. Other driver cited for following too closely. "
    "Estimated vehicle repair: $8,500. Medical bills to date: $3,200."
)

print(f"\nClaim: {test_claim[:80]}...\n")

result = dispatcher.execute(TaskType.CLASSIFY, test_claim)
print(f"Task: classify")
print(f"Response: {result['text']}")
print(f"Tokens: {result['input_tokens']} in / {result['output_tokens']} out")
print(f"Latency: {result['latency']}s")

if result.get('parsed'):
    print(f"\nParsed JSON:")
    for k, v in result['parsed'].items():
        print(f"  {k}: {v}")

print(f"\n{'=' * 65}")
print("Registry loaded. Dispatcher tested. Ready for pipeline in Cell 2.")
print(f"{'=' * 65}")

Day 29: Multi-Task Architecture
Task Registry & Dispatcher

Registered Tasks: 5

  classify        │ Claim Classification         │ format=json │ depends=['none']
  severity        │ Severity Assessment          │ format=json │ depends=['classify']
  fraud_screen    │ Fraud Screening              │ format=json │ depends=['classify']
  coverage        │ Coverage Determination       │ format=json │ depends=['classify', 'severity', 'fraud_screen']
  communicate     │ Policyholder Communication   │ format=text │ depends=['classify', 'coverage']

DISPATCHER TEST: Single Task (Classification)

Claim: Rear-end collision on Highway 401. Three vehicles involved. Claimant reports whi...

Task: classify
Response: ```json
{
  "claim_type": "auto",
  "sub_type": "collision with bodily injury",
  "confidence": "HIGH",
  "reasoning": "Vehicle collision on highway with property damage to automobile and personal injury (whiplash), clearly falling under auto insurance coverage for both physical damage a

In [2]:
# =============================================================
# Cell 2 — Multi-Task Pipeline with Shared Context
# =============================================================
# The dispatcher handles individual tasks. The pipeline orchestrates
# them in sequence, passing each task's output as context to the
# next. This is the "assembly line" — predefined order, growing
# context at each station.
# =============================================================

class ClaimsPipeline:
    """Orchestrates multiple tasks in sequence with shared context."""
    
    def __init__(self, dispatcher: TaskDispatcher):
        self.dispatcher = dispatcher
        self.context: Dict[TaskType, Dict] = {}  # shared state
        self.execution_log: List[Dict] = []       # audit trail
        self.claim_text: str = ''
    
    def reset(self):
        """Clear state for a new claim."""
        self.context = {}
        self.execution_log = []
        self.claim_text = ''
    
    def run_task(self, task_type: TaskType, verbose=True) -> Dict:
        """Execute a single task within the pipeline context."""
        config = self.dispatcher.get_task(task_type)
        
        if verbose:
            deps = [d.value for d in config.depends_on] if config.depends_on else ['none']
            print(f"\n  ┌─ {config.name} ({task_type.value})")
            print(f"  │  Dependencies: {deps}")
            print(f"  │  Context keys: {[k.value for k in self.context.keys()]}")
        
        result = self.dispatcher.execute(
            task_type, 
            self.claim_text, 
            self.context
        )
        
        # Store result in shared context
        self.context[task_type] = result
        
        # Log for audit
        self.execution_log.append({
            'task': task_type.value,
            'name': config.name,
            'input_tokens': result['input_tokens'],
            'output_tokens': result['output_tokens'],
            'latency': result['latency'],
            'success': 'error' not in result,
            'parsed': result.get('parsed') is not None
        })
        
        if verbose:
            print(f"  │  Tokens: {result['input_tokens']} in / {result['output_tokens']} out")
            print(f"  │  Latency: {result['latency']}s")
            if result.get('parsed'):
                # Show key fields from parsed JSON
                parsed = result['parsed']
                preview_fields = list(parsed.items())[:3]
                for k, v in preview_fields:
                    val_str = str(v)[:60]
                    print(f"  │  → {k}: {val_str}")
            else:
                print(f"  │  → {result['text'][:100]}...")
            print(f"  └─ ✓ Complete")
        
        return result
    
    def run_sequence(self, task_types: List[TaskType], claim_text: str, 
                     verbose=True) -> Dict[TaskType, Dict]:
        """Run a predefined sequence of tasks on a claim."""
        self.reset()
        self.claim_text = claim_text
        
        if verbose:
            print(f"\n{'=' * 65}")
            print(f"PIPELINE: {len(task_types)} tasks")
            print(f"Sequence: {' → '.join(t.value for t in task_types)}")
            print(f"{'=' * 65}")
            print(f"\nClaim: {claim_text[:100]}...")
        
        for task_type in task_types:
            self.run_task(task_type, verbose=verbose)
        
        if verbose:
            self._print_summary()
        
        return self.context
    
    def _print_summary(self):
        """Print pipeline execution summary."""
        total_in = sum(e['input_tokens'] for e in self.execution_log)
        total_out = sum(e['output_tokens'] for e in self.execution_log)
        total_latency = sum(e['latency'] for e in self.execution_log)
        total_cost = (total_in * 3 + total_out * 15) / 1_000_000
        
        print(f"\n{'─' * 65}")
        print(f"PIPELINE SUMMARY")
        print(f"{'─' * 65}")
        print(f"  Tasks completed:  {len(self.execution_log)}")
        print(f"  Total tokens:     {total_in + total_out:,} "
              f"({total_in:,} in / {total_out:,} out)")
        print(f"  Total latency:    {total_latency:.1f}s")
        print(f"  Total cost:       ${total_cost:.4f}")
        print(f"  Context keys:     {[k.value for k in self.context.keys()]}")
    
    def get_context_snapshot(self) -> str:
        """Return a readable view of all accumulated context."""
        snapshot = []
        for task_type, result in self.context.items():
            snapshot.append(f"\n{'─' * 40}")
            snapshot.append(f"[{task_type.value.upper()}]")
            if result.get('parsed'):
                snapshot.append(json.dumps(result['parsed'], indent=2))
            else:
                snapshot.append(result['text'])
        return '\n'.join(snapshot)


# ----- Define Standard Pipelines -----
# These are our "assembly lines" — predefined task sequences

STANDARD_PIPELINE = [
    TaskType.CLASSIFY,
    TaskType.SEVERITY,
    TaskType.FRAUD_SCREEN,
    TaskType.COVERAGE,
    TaskType.COMMUNICATE
]

FAST_TRIAGE = [
    TaskType.CLASSIFY,
    TaskType.SEVERITY
]

FRAUD_FOCUSED = [
    TaskType.CLASSIFY,
    TaskType.FRAUD_SCREEN
]


# ----- Run the Standard Pipeline -----
print("=" * 65)
print("Day 29: Multi-Task Pipeline with Shared Context")
print("=" * 65)

pipeline = ClaimsPipeline(dispatcher)

# Use the same test claim from Cell 1
results = pipeline.run_sequence(STANDARD_PIPELINE, test_claim)

# ----- Show the accumulated context -----
print(f"\n\n{'=' * 65}")
print("FULL CONTEXT SNAPSHOT")
print("=" * 65)
print(pipeline.get_context_snapshot())

# ----- Show how context grew at each step -----
print(f"\n\n{'=' * 65}")
print("CONTEXT GROWTH: What each task could see")
print("=" * 65)
for i, task_type in enumerate(STANDARD_PIPELINE):
    config = dispatcher.get_task(task_type)
    deps = [d.value for d in config.depends_on] if config.depends_on else ['none']
    context_size = sum(
        len(results[d]['text']) for d in config.depends_on 
        if d in results
    )
    print(f"\n  Step {i+1}: {task_type.value}")
    print(f"    Saw context from: {deps}")
    print(f"    Context size: ~{context_size} chars")

Day 29: Multi-Task Pipeline with Shared Context

PIPELINE: 5 tasks
Sequence: classify → severity → fraud_screen → coverage → communicate

Claim: Rear-end collision on Highway 401. Three vehicles involved. Claimant reports whiplash and vehicle da...

  ┌─ Claim Classification (classify)
  │  Dependencies: ['none']
  │  Context keys: []
  │  Tokens: 155 in / 82 out
  │  Latency: 2.54s
  │  → claim_type: auto
  │  → sub_type: collision with bodily injury
  │  → confidence: HIGH
  └─ ✓ Complete

  ┌─ Severity Assessment (severity)
  │  Dependencies: ['classify']
  │  Context keys: ['classify']
  │  Tokens: 235 in / 165 out
  │  Latency: 4.33s
  │  → severity: MEDIUM
  │  → priority_score: 5
  │  → factors: ['Soft tissue injury (whiplash) with moderate medical treatm
  └─ ✓ Complete

  ┌─ Fraud Screening (fraud_screen)
  │  Dependencies: ['classify']
  │  Context keys: ['classify', 'severity']
  │  Tokens: 268 in / 215 out
  │  Latency: 5.31s
  │  → fraud_risk: LOW
  │  → risk_score: 2
  │ 

In [ ]:
# =============================================================
# Cell 3 — End-to-End Claims Processing
# =============================================================
# Run multiple claims through different pipelines to see:
# 1. How shared context improves downstream task quality
# 2. How different pipelines serve different business needs
# 3. Cost differences between full and abbreviated pipelines
# =============================================================

# ----- Test Claims -----
PIPELINE_CLAIMS = [
    {
        'id': 'CLM-201',
        'description': (
            'Single-vehicle accident. Claimant hit a deer on rural road at dusk. '
            'Front-end damage including hood, grille, headlights, and radiator. '
            'Vehicle towed from scene. No injuries. Comprehensive coverage applies. '
            'Estimated repair: $6,200. Vehicle value: $18,000.'
        ),
        'expected_complexity': 'LOW'
    },
    {
        'id': 'CLM-202',
        'description': (
            'Multi-vehicle accident on Highway 401. Insured rear-ended by '
            'third party who left the scene. Insured then pushed into vehicle ahead. '
            'Insured reports neck pain and headaches. Passenger taken by ambulance. '
            'Police report filed. Other driver cited at fault. Estimated vehicle '
            'damage: $22,000. Medical bills: $15,000 and climbing. '
            'Vehicle may be total loss.'
        ),
        'expected_complexity': 'HIGH'
    },
    {
        'id': 'CLM-203',
        'description': (
            'Insured reports jewelry stolen from home. Claims a $35,000 watch '
            'and $12,000 necklace were taken. Discovered missing after returning '
            'from two-week vacation. No signs of forced entry. No police report '
            'filed. Alarm system was not activated. No witnesses. '
            'Policy limit: $400,000. Deductible: $2,500.'
        ),
        'expected_complexity': 'HIGH — fraud indicators'
    }
]


# ----- Run each claim through the STANDARD pipeline -----
print("=" * 65)
print("STANDARD PIPELINE: All 5 tasks, full context flow")
print("=" * 65)

pipeline_results = {}

for claim in PIPELINE_CLAIMS:
    print(f"\n\n{'━' * 65}")
    print(f"CLAIM: {claim['id']} (Expected: {claim['expected_complexity']})")
    print(f"{'━' * 65}")
    print(f"{claim['description'][:100]}...")
    
    pipe = ClaimsPipeline(dispatcher)
    results = pipe.run_sequence(STANDARD_PIPELINE, claim['description'])
    pipeline_results[claim['id']] = {
        'pipeline': pipe,
        'results': results
    }


# ----- Compare: Full Pipeline vs Fast Triage on the same claim -----
print(f"\n\n{'=' * 65}")
print("PIPELINE COMPARISON: Standard vs Fast Triage vs Fraud Focused")
print(f"{'=' * 65}")
print(f"\nUsing claim CLM-202 (complex multi-vehicle accident)\n")

comparison_claim = PIPELINE_CLAIMS[1]['description']

pipelines_to_compare = [
    ('Standard (5 tasks)',     STANDARD_PIPELINE),
    ('Fast Triage (2 tasks)',  FAST_TRIAGE),
    ('Fraud Focused (2 tasks)', FRAUD_FOCUSED),
]

comparison_results = {}

for name, task_sequence in pipelines_to_compare:
    print(f"\n{'─' * 65}")
    print(f"Pipeline: {name}")
    print(f"Sequence: {' → '.join(t.value for t in task_sequence)}")
    print(f"{'─' * 65}")
    
    pipe = ClaimsPipeline(dispatcher)
    results = pipe.run_sequence(task_sequence, comparison_claim, verbose=True)
    
    total_in = sum(e['input_tokens'] for e in pipe.execution_log)
    total_out = sum(e['output_tokens'] for e in pipe.execution_log)
    total_latency = sum(e['latency'] for e in pipe.execution_log)
    total_cost = (total_in * 3 + total_out * 15) / 1_000_000
    
    comparison_results[name] = {
        'tasks': len(task_sequence),
        'total_tokens': total_in + total_out,
        'input_tokens': total_in,
        'output_tokens': total_out,
        'latency': total_latency,
        'cost': total_cost,
        'context_keys': [k.value for k in results.keys()]
    }


# ----- Comparison Dashboard -----
print(f"\n\n{'=' * 65}")
print("PIPELINE COMPARISON DASHBOARD")
print(f"{'=' * 65}")

print(f"\n{'Pipeline':<28} {'Tasks':>5} {'Tokens':>8} {'Latency':>9} {'Cost':>10}")
print(f"{'-'*28} {'-'*5} {'-'*8} {'-'*9} {'-'*10}")

for name, stats in comparison_results.items():
    print(f"{name:<28} {stats['tasks']:>5} {stats['total_tokens']:>8,} "
          f"{stats['latency']:>7.1f}s ${stats['cost']:>8.4f}")

# Cost at scale
print(f"\n{'Pipeline':<28} {'$/10K Claims':>14} {'$/100K Claims':>15}")
print(f"{'-'*28} {'-'*14} {'-'*15}")
for name, stats in comparison_results.items():
    cost_10k = stats['cost'] * 10000
    cost_100k = stats['cost'] * 100000
    print(f"{name:<28} ${cost_10k:>12,.2f} ${cost_100k:>13,.2f}")


# ----- Context Impact Analysis -----
print(f"\n\n{'=' * 65}")
print("CONTEXT IMPACT: How shared context changes fraud detection")
print(f"{'=' * 65}")
print(f"\nClaim CLM-203: The suspicious jewelry theft\n")

# Run fraud screen WITHOUT context (standalone)
print("--- Fraud Screen: WITHOUT prior context ---")
standalone_pipe = ClaimsPipeline(dispatcher)
standalone_pipe.claim_text = PIPELINE_CLAIMS[2]['description']
# Run classify first to satisfy dependency but don't show it
standalone_pipe.run_task(TaskType.CLASSIFY, verbose=False)
# Now run fraud with only classify context
standalone_result = standalone_pipe.run_task(TaskType.FRAUD_SCREEN, verbose=False)

print(f"Context available: [classify only]")
if standalone_result.get('parsed'):
    fr = standalone_result['parsed']
    print(f"  Fraud risk: {fr.get('fraud_risk', 'N/A')}")
    print(f"  Risk score: {fr.get('risk_score', 'N/A')}/10")
    print(f"  Recommendation: {fr.get('recommendation', 'N/A')}")
    if fr.get('indicators'):
        print(f"  Indicators ({len(fr['indicators'])}):")
        for ind in fr['indicators']:
            print(f"    • {ind}")

# Now get the fraud result from the FULL pipeline run
print(f"\n--- Fraud Screen: WITH full pipeline context ---")
full_fraud = pipeline_results['CLM-203']['results'].get(TaskType.FRAUD_SCREEN)

if full_fraud and full_fraud.get('parsed'):
    fr = full_fraud['parsed']
    print(f"Context available: [classify + severity]")
    print(f"  Fraud risk: {fr.get('fraud_risk', 'N/A')}")
    print(f"  Risk score: {fr.get('risk_score', 'N/A')}/10")
    print(f"  Recommendation: {fr.get('recommendation', 'N/A')}")
    if fr.get('indicators'):
        print(f"  Indicators ({len(fr['indicators'])}):")
        for ind in fr['indicators']:
            print(f"    • {ind}")
elif full_fraud:
    print(f"  Raw response: {full_fraud['text'][:300]}")


# ----- Communication Output -----
print(f"\n\n{'=' * 65}")
print("FINAL OUTPUT: Policyholder Communication (CLM-201)")
print(f"{'=' * 65}")

comm_result = pipeline_results['CLM-201']['results'].get(TaskType.COMMUNICATE)
if comm_result:
    print(f"\n{comm_result['text']}")
    print(f"\n[This email was informed by: classify → severity → "
          f"fraud_screen → coverage]")

print(f"\n\n{'=' * 65}")
print("Pipeline processing complete. Ready for metrics in Cell 4.")
print(f"{'=' * 65}")

STANDARD PIPELINE: All 5 tasks, full context flow


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CLAIM: CLM-201 (Expected: LOW)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Single-vehicle accident. Claimant hit a deer on rural road at dusk. Front-end damage including hood,...

PIPELINE: 5 tasks
Sequence: classify → severity → fraud_screen → coverage → communicate

Claim: Single-vehicle accident. Claimant hit a deer on rural road at dusk. Front-end damage including hood,...

  ┌─ Claim Classification (classify)
  │  Dependencies: ['none']
  │  Context keys: []
  │  Tokens: 159 in / 66 out
  │  Latency: 2.2s
  │  → claim_type: auto
  │  → sub_type: comprehensive
  │  → confidence: HIGH
  └─ ✓ Complete

  ┌─ Severity Assessment (severity)
  │  Dependencies: ['classify']
  │  Context keys: ['classify']
  │  Tokens: 223 in / 127 out
  │  Latency: 3.21s
  │  → severity: LOW
  │  → priority_score: 2
  │  → factors: ['No bodily injury reported', 'Singl

In [ ]:
# =============================================================
# Cell 4 — Metrics & Production Strategy
# =============================================================
# Analyze pipeline performance across all runs, calculate
# production cost projections, and define the operational
# strategy for deploying multi-task architecture.
# =============================================================

# ----- Collect metrics from all pipeline runs -----
print("=" * 65)
print("PIPELINE METRICS ANALYSIS")
print("=" * 65)

# Gather per-task metrics across all standard pipeline runs
task_metrics: Dict[str, List[Dict]] = {}

for claim_id, data in pipeline_results.items():
    pipe = data['pipeline']
    for entry in pipe.execution_log:
        task_name = entry['task']
        if task_name not in task_metrics:
            task_metrics[task_name] = []
        task_metrics[task_name].append({
            'claim_id': claim_id,
            'input_tokens': entry['input_tokens'],
            'output_tokens': entry['output_tokens'],
            'latency': entry['latency'],
            'cost': (entry['input_tokens'] * 3 + entry['output_tokens'] * 15) / 1_000_000
        })

print(f"\n{'Task':<15} {'Avg In':>8} {'Avg Out':>8} {'Avg Lat':>9} "
      f"{'Avg Cost':>10} {'% of Pipeline':>14}")
print("-" * 68)

total_pipeline_cost = sum(
    m['cost'] for metrics in task_metrics.values() for m in metrics
) / len(PIPELINE_CLAIMS)

for task_name in ['classify', 'severity', 'fraud_screen', 'coverage', 'communicate']:
    if task_name in task_metrics:
        metrics = task_metrics[task_name]
        avg_in = sum(m['input_tokens'] for m in metrics) / len(metrics)
        avg_out = sum(m['output_tokens'] for m in metrics) / len(metrics)
        avg_lat = sum(m['latency'] for m in metrics) / len(metrics)
        avg_cost = sum(m['cost'] for m in metrics) / len(metrics)
        pct = (avg_cost / total_pipeline_cost * 100) if total_pipeline_cost > 0 else 0
        
        print(f"  {task_name:<13} {avg_in:>8.0f} {avg_out:>8.0f} "
              f"{avg_lat:>7.1f}s ${avg_cost:>8.4f} {pct:>12.1f}%")

print(f"  {'─'*13} {'─'*8} {'─'*8} {'─'*9} {'─'*10} {'─'*14}")
print(f"  {'TOTAL':<13} {'':>8} {'':>8} {'':>9} "
      f"${total_pipeline_cost:>8.4f} {'100.0%':>14}")


# ----- Token Growth Analysis: How context inflates input tokens -----
print(f"\n\n{'=' * 65}")
print("CONTEXT GROWTH: Input token inflation across pipeline")
print(f"{'=' * 65}")
print(f"\nAs context accumulates, later tasks receive more input tokens.")
print(f"This is the cost of shared context — each task sees more.\n")

for claim_id, data in pipeline_results.items():
    pipe = data['pipeline']
    print(f"\n  {claim_id}:")
    cumulative_context = 0
    for entry in pipe.execution_log:
        tokens_in = entry['input_tokens']
        bar = '█' * (tokens_in // 20)
        print(f"    {entry['task']:<15} {tokens_in:>5} input tokens  {bar}")
        cumulative_context += entry['output_tokens']


# ----- Pipeline Selection Strategy -----
print(f"\n\n{'=' * 65}")
print("PIPELINE SELECTION STRATEGY")
print(f"{'=' * 65}")

print(f"""
Not every claim needs the full 5-task pipeline. Choosing the right
pipeline based on the claim's characteristics saves cost without
sacrificing quality where it matters.

┌──────────────────┬────────────────────────────────────────────────┐
│ Pipeline         │ When to Use                                    │
├──────────────────┼────────────────────────────────────────────────┤
│ FAST TRIAGE      │ Initial intake: classify + severity only       │
│ (2 tasks)        │ → Route LOW severity to automated processing   │
│                  │ → Route MEDIUM/HIGH to appropriate pipeline    │
├──────────────────┼────────────────────────────────────────────────┤
│ FRAUD FOCUSED    │ Claims with fraud indicators detected in       │
│ (2 tasks)        │ triage or flagged by rules engine              │
│                  │ → Deep fraud analysis before proceeding        │
├──────────────────┼────────────────────────────────────────────────┤
│ STANDARD         │ All claims that pass triage + fraud screen     │
│ (5 tasks)        │ → Full analysis for adjuster review            │
│                  │ → Generates policyholder communication         │
└──────────────────┴────────────────────────────────────────────────┘
""")


# ----- Cost Projections -----
print(f"{'=' * 65}")
print("COST PROJECTIONS: Monthly Volume Scenarios")
print(f"{'=' * 65}")

# Assume a realistic distribution of claim complexity
# Based on industry norms: 60% simple, 25% medium, 15% complex
print(f"\nAssumption: 60% Fast Triage only, 25% Standard, 15% Fraud Focused")
print(f"(Based on typical P&C claims distribution)\n")

# Get average costs per pipeline from comparison results
fast_cost = comparison_results.get('Fast Triage (2 tasks)', {}).get('cost', 0.005)
standard_cost = comparison_results.get('Standard (5 tasks)', {}).get('cost', 0.03)
fraud_cost = comparison_results.get('Fraud Focused (2 tasks)', {}).get('cost', 0.008)

volumes = [5000, 10000, 25000, 50000, 100000]

print(f"{'Monthly Volume':>15} {'Fast (60%)':>12} {'Standard (25%)':>15} "
      f"{'Fraud (15%)':>13} {'Total':>10} {'All-Standard':>14}")
print("-" * 82)

for vol in volumes:
    fast_total = vol * 0.60 * fast_cost
    std_total = vol * 0.25 * standard_cost
    fraud_total = vol * 0.15 * fraud_cost
    mixed_total = fast_total + std_total + fraud_total
    all_standard = vol * standard_cost
    savings_pct = (all_standard - mixed_total) / all_standard * 100 if all_standard > 0 else 0
    
    print(f"{vol:>15,} ${fast_total:>10,.2f} ${std_total:>13,.2f} "
          f"${fraud_total:>11,.2f} ${mixed_total:>8,.2f} "
          f"${all_standard:>12,.2f}")

print(f"\n  Mixed routing saves ~{savings_pct:.0f}% vs running every claim through Standard pipeline")


# ----- Production Architecture -----
print(f"\n\n{'=' * 65}")
print("PRODUCTION ARCHITECTURE")
print(f"{'=' * 65}")

print(f"""
                    ┌──────────────┐
                    │  Claim Intake │
                    └──────┬───────┘
                           │
                    ┌──────▼───────┐
                    │  FAST TRIAGE │
                    │  classify +  │
                    │  severity    │
                    └──────┬───────┘
                           │
              ┌────────────┼────────────┐
              │            │            │
        ┌─────▼─────┐ ┌───▼────┐ ┌────▼─────┐
        │ LOW sev.  │ │ MEDIUM │ │ HIGH sev.│
        │ Auto-     │ │ Std.   │ │ Fraud +  │
        │ process   │ │ Pipeline│ │ Standard │
        └─────┬─────┘ └───┬────┘ └────┬─────┘
              │            │            │
              │       ┌────▼────┐  ┌───▼──────┐
              │       │coverage │  │fraud     │
              │       │+comms   │  │screen    │
              │       └────┬────┘  └───┬──────┘
              │            │       ┌───▼──────┐
              │            │       │coverage  │
              │            │       │+comms    │
              │            │       └───┬──────┘
              │            │           │
              └────────────┼───────────┘
                           │
                    ┌──────▼───────┐
                    │  Adjuster    │
                    │  Queue       │
                    └──────────────┘

Key Decisions:
─────────────────────────────────────────────────────────
1. Every claim gets Fast Triage — it's cheap and fast
2. Severity determines the downstream pipeline
3. Fraud screening only runs when warranted
4. Communication always generated at the end
5. All results stored in shared context for audit trail

Monitoring (CloudWatch):
─────────────────────────────────────────────────────────
- Pipeline distribution: % claims per pipeline type
- Task latency: P50, P95, P99 per task
- Token usage: daily/weekly trend
- Parse errors: JSON failures per task
- Cost per claim: rolling 7-day average
- Fraud hit rate: % flagged INVESTIGATE
""")


# ----- Next Steps: The Bridge to Agents -----
print(f"{'=' * 65}")
print("THE BRIDGE TO AGENTS")
print(f"{'=' * 65}")

print(f"""
What we built today is a STATIC pipeline — you define the sequence,
the claims flow through it. This works well and is production-ready.

But notice the limitations:
  • Every HIGH claim gets the same treatment
  • The pipeline can't skip a step if it's not needed
  • If fraud screen returns INVESTIGATE, we still send a
    standard policyholder email (we probably shouldn't)

An AGENT solves this by making the routing REACTIVE:

  Static Pipeline (Day 29):
    classify → severity → fraud → coverage → communicate
    (always the same sequence)

  Agent (Phases 10-11):
    classify → [LLM decides] → fraud → [LLM decides] → escalate to SIU
    (sequence emerges from the claim)

What stays the same:
  ✓ Task registry (same tasks, same prompts)
  ✓ Shared context (same state mechanism)
  ✓ Dispatcher (same execution logic)

What changes:
  ✗ Hardcoded sequence → LLM-driven routing
  ✗ Fixed pipeline → Dynamic decision loop
  ✗ Predefined exit → Goal-based termination

The architecture we built today IS the agent's toolkit.
Phases 10-11 add the reasoning loop.
""")